# RDR2 Sentiment Analysis — Sentiment Scoring & Entity Matching

Takes the raw text collected in `01_data_collection.ipynb` and produces the aggregated, entity-level sentiment tables used throughout the report.

**Pipeline steps:**
1. Score every review/comment with VADER, using a custom lexicon tuned for gaming slang
2. Match text to characters, missions, and chapters via curated regex patterns (with two collision fixes discovered along the way — documented inline)
3. Aggregate sentiment per entity, with a mention-count reliability threshold
4. Compute sentiment-over-time trends and the "epilogue effect"
5. Attempt a patch-date correlation analysis (kept for documentation — ultimately excluded from the final report due to small sample sizes)
6. Compute RDR2 vs. GTA V comparison metrics

**Note on this cleaned version:** several one-off debug/print cells and superseded intermediate attempts from the original working notebook have been removed for readability. Where a fix was made in response to a discovered issue (e.g. regex collisions), the diagnostic cell that revealed the issue is kept, so the reasoning stays visible — only pure exploratory prints with no bearing on the final pipeline were dropped.

**Depends on:** all CSVs produced by `01_data_collection.ipynb`.


**Note on file paths:** this notebook lives in `notebooks/`, so paths point one level up (`../data/...`). The raw Steam reviews and YouTube comments (`data/raw/`) are **not included in the repository**, because they contain other users' review texts and identifiers; rerun `01_data_collection.ipynb` with your own API keys to recreate them.

## Setup
Load all raw text sources and reference/lookup tables.

In [ ]:
import pandas as pd

# Load raw text sources
steam_recent = pd.read_csv("../data/raw/steam_reviews_recent.csv")
steam_historical = pd.read_csv("../data/raw/steam_reviews_historical.csv")
youtube_comments = pd.read_csv("../data/raw/youtube_comments_raw.csv")
youtube_video_map = pd.read_csv("../data/processed/youtube_video_entity_map.csv")
youtube_videos = pd.read_csv("../data/processed/youtube_videos_raw.csv")

# Load reference lists
ref_chapters = pd.read_csv("../data/reference/reference_chapters.csv")
ref_missions = pd.read_csv("../data/reference/reference_missions.csv")
ref_characters = pd.read_csv("../data/reference/reference_characters.csv")

print("Steam recent:", steam_recent.shape)
print("Steam historical:", steam_historical.shape)
print("YouTube comments:", youtube_comments.shape)
print("YouTube videos:", youtube_videos.shape)
print("Reference missions:", ref_missions.shape)
print("Reference characters:", ref_characters.shape)

# Reliability threshold used throughout: entities with fewer mentions than this
# are flagged low_confidence rather than excluded, so small-sample noise (e.g.
# a character with 5 mentions and one very negative outlier) doesn't get
# reported with the same confidence as a well-sampled entity.
MIN_MENTION_THRESHOLD = 20

## Sentiment scoring
Install VADER, the lexicon-based sentiment analyzer used throughout this project.

In [ ]:
pip install vaderSentiment --break-system-packages

**Baseline test** — before any customization, VADER misreads common gaming slang. Words like "destroyed" and "wrecked" are strongly negative in general English, but in game reviews they usually signal strong *positive* emotional impact ("this game destroyed me" = deeply moving). This test illustrates the problem the custom lexicon below solves.

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

test_sentences = [
    "This game destroyed me emotionally, best story ever.",
    "Terrible optimization, constant crashes, waste of money.",
    "It's fine, nothing special."
]

for s in test_sentences:
    scores = analyzer.polarity_scores(s)
    print(f"{s}\n -> {scores}\n")

**Final scoring** — seven terms overridden to positive, based on common gaming-review usage (see baseline test above for why this is necessary). Applied across all three text sources and saved back to their source CSVs.

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

# Targeted overrides for gaming/emotional-impact slang common in RDR2 discourse.
# Rationale: these words are typically negative in general English, but in gaming
# reviews/comments they often signal strong positive emotional impact.
gaming_slang_overrides = {
    "destroyed": 2.5,    # "this game destroyed me" = deeply moved, positive
    "wrecked": 2.3,      # "the ending wrecked me"
    "ruined": 1.8,       # "ruined me emotionally" - context-dependent, moderate positive
    "broke": 1.5,        # "broke me" - emotional impact
    "gutted": 2.0,       # "I was gutted" - British slang for deeply affected
    "shattered": 1.8,    # emotional impact
    "obliterated": 2.0,  # similar emotional-impact usage
}

analyzer.lexicon.update(gaming_slang_overrides)

def get_sentiment(text):
    """Return VADER's compound sentiment score for a piece of text, or None if empty/invalid."""
    if pd.isna(text) or not isinstance(text, str) or text.strip() == "":
        return None
    return analyzer.polarity_scores(text)["compound"]

# Apply across all three text sources
steam_recent["sentiment"] = steam_recent["review_text"].apply(get_sentiment)
steam_historical["sentiment"] = steam_historical["review_text"].apply(get_sentiment)
youtube_comments["sentiment"] = youtube_comments["comment_text"].apply(get_sentiment)

print("Steam recent sentiment:")
print(steam_recent["sentiment"].describe())
print("\nSteam historical sentiment:")
print(steam_historical["sentiment"].describe())
print("\nYouTube comments sentiment:")
print(youtube_comments["sentiment"].describe())

In [ ]:
steam_recent.to_csv("../data/raw/steam_reviews_recent.csv", index=False)
steam_historical.to_csv("../data/raw/steam_reviews_historical.csv", index=False)
youtube_comments.to_csv("../data/raw/youtube_comments_raw.csv", index=False)
print("Sentiment scores saved to all three files.")

## Entity matching: missions & characters

Regex-based matching against the curated reference lists, using word boundaries to avoid partial-word false matches (e.g. "Bill" matching inside "billboard").

In [ ]:
import re

def build_pattern_dict(df, name_col, alt_col):
    """
    Builds a dict of {entity_name: compiled_regex} from a reference dataframe.
    Combines the main name + all alt-phrases into one pattern per entity,
    using word boundaries to avoid partial-word false matches (e.g. Bill vs billboard).
    """
    patterns = {}
    for _, row in df.iterrows():
        entity = row[name_col]
        terms = [row[name_col]] + [t.strip() for t in str(row[alt_col]).split(",")]
        terms = [t for t in terms if t and t.lower() != "nan"]
        # Escape special regex characters, join with word boundaries
        escaped = [re.escape(t) for t in terms]
        pattern = r"\b(" + "|".join(escaped) + r")\b"
        patterns[entity] = re.compile(pattern, re.IGNORECASE)
    return patterns


mission_patterns = build_pattern_dict(ref_missions, "mission_name", "alt_phrases")
character_patterns = build_pattern_dict(ref_characters, "character_name", "alt_names")


def find_matches(text, pattern_dict):
    """
    Returns a list of entity names whose pattern matches somewhere in the text.
    A single text can match multiple entities.
    """
    if pd.isna(text) or not isinstance(text, str):
        return []
    matches = []
    for entity, pattern in pattern_dict.items():
        if pattern.search(text):
            matches.append(entity)
    return matches


# Apply matching to Steam reviews (recent + historical) and YouTube comments
for df, text_col in [(steam_recent, "review_text"), (steam_historical, "review_text"), (youtube_comments, "comment_text")]:
    df["matched_missions"] = df[text_col].apply(lambda t: find_matches(t, mission_patterns))
    df["matched_characters"] = df[text_col].apply(lambda t: find_matches(t, character_patterns))

# Quick check: how many rows actually matched something
print("Steam recent - rows with mission match:", (steam_recent["matched_missions"].str.len() > 0).sum())
print("Steam recent - rows with character match:", (steam_recent["matched_characters"].str.len() > 0).sum())
print("YouTube comments - rows with mission match:", (youtube_comments["matched_missions"].str.len() > 0).sum())
print("YouTube comments - rows with character match:", (youtube_comments["matched_characters"].str.len() > 0).sum())

**Collision found:** two mission names ("Red Dead Redemption" and "Red Dead Redemption II") are also the game's own title, so any review that just mentions the game by name was being counted as a mission mention — 601 false positives before the fix (see the mission-matching fix further below).

In [ ]:
# Check how many mission matches include the ambiguous title
overlap_check = steam_recent[steam_recent["matched_missions"].apply(lambda x: "Red Dead Redemption" in x and "Red Dead Redemption II" in x)]
print("Rows matching both RDR and RDR2 mission titles:", len(overlap_check))
if len(overlap_check) > 0:
    print(overlap_check[["review_text", "matched_missions"]].head(3))

## Character sentiment aggregation

Aggregates sentiment per character across all three text sources, computing two distinct polarization metrics:
- **`polarization_raw`** (mention_count × sentiment_std) — captures characters who are both widely discussed *and* divisive
- **`polarization_per_mention`** (sentiment_std alone) — captures divisiveness independent of volume, but is more exposed to small-sample noise

Entities below `MIN_MENTION_THRESHOLD` are flagged `low_confidence`, not excluded — downstream analysis (and the report) chooses when to filter on this flag.

In [ ]:
def aggregate_entity_sentiment(dfs_with_cols, match_col, entity_names):
    """
    dfs_with_cols: list of (dataframe, sentiment_col) tuples to pull sentiment from
    match_col: column containing list of matched entities (e.g. 'matched_characters')
    entity_names: list of all entity names to report on, even if zero mentions
    """
    records = []
    for entity in entity_names:
        sentiments = []
        for df, sentiment_col in dfs_with_cols:
            mask = df[match_col].apply(lambda x: entity in x)
            sentiments.extend(df.loc[mask, sentiment_col].dropna().tolist())

        if sentiments:
            records.append({
                "entity_name": entity,
                "mention_count": len(sentiments),
                "avg_sentiment": pd.Series(sentiments).mean(),
                "sentiment_std": pd.Series(sentiments).std(),
                "min_sentiment": min(sentiments),
                "max_sentiment": max(sentiments)
            })
        else:
            records.append({
                "entity_name": entity,
                "mention_count": 0,
                "avg_sentiment": None,
                "sentiment_std": None,
                "min_sentiment": None,
                "max_sentiment": None
            })
    return pd.DataFrame(records)


character_names = ref_characters["character_name"].tolist()

df_char_sentiment = aggregate_entity_sentiment(
    dfs_with_cols=[
        (steam_recent, "sentiment"),
        (steam_historical, "sentiment"),
        (youtube_comments, "sentiment")
    ],
    match_col="matched_characters",
    entity_names=character_names
)

df_char_sentiment["polarization_raw"] = df_char_sentiment["mention_count"] * df_char_sentiment["sentiment_std"]
df_char_sentiment["polarization_per_mention"] = df_char_sentiment["sentiment_std"]
df_char_sentiment["low_confidence"] = df_char_sentiment["mention_count"] < MIN_MENTION_THRESHOLD

df_char_sentiment = df_char_sentiment.sort_values("mention_count", ascending=False)
df_char_sentiment.to_csv("../data/processed/characters_sentiment.csv", index=False)
df_char_sentiment

## Mission sentiment aggregation

**Fix applied before aggregating:** for the two title-colliding entries ("Red Dead Redemption" and "Red Dead Redemption II"), only their alt-phrases are used for matching — not the mission name itself — since the mission name is identical to the game's title and was causing the 601 false positives found above. This dropped false positives to 2.

In [ ]:
def build_mission_patterns_fixed(df, name_col, alt_col):
    """Same as build_pattern_dict, but two entries whose name collides with the
    game's own title are matched on alt-phrases only, not the name itself."""
    patterns = {}
    for _, row in df.iterrows():
        entity = row[name_col]

        # These two entries collide with the game's own title - use alt_phrases only
        if entity in ["Red Dead Redemption", "Red Dead Redemption II"]:
            terms = [t.strip() for t in str(row[alt_col]).split(",")]
        else:
            terms = [row[name_col]] + [t.strip() for t in str(row[alt_col]).split(",")]

        terms = [t for t in terms if t and t.lower() != "nan"]
        escaped = [re.escape(t) for t in terms]
        pattern = r"\b(" + "|".join(escaped) + r")\b"
        patterns[entity] = re.compile(pattern, re.IGNORECASE)
    return patterns


mission_patterns = build_mission_patterns_fixed(ref_missions, "mission_name", "alt_phrases")

# Re-run matching on all three sources with the fixed patterns
for df, text_col in [(steam_recent, "review_text"), (steam_historical, "review_text"), (youtube_comments, "comment_text")]:
    df["matched_missions"] = df[text_col].apply(lambda t: find_matches(t, mission_patterns))

print("Red Dead Redemption (ending) matches now:", steam_recent["matched_missions"].apply(lambda x: "Red Dead Redemption" in x).sum())
print("Red Dead Redemption II (finale) matches now:", steam_recent["matched_missions"].apply(lambda x: "Red Dead Redemption II" in x).sum())

In [ ]:
mission_names = ref_missions["mission_name"].tolist()

df_mission_sentiment = aggregate_entity_sentiment(
    dfs_with_cols=[
        (steam_recent, "sentiment"),
        (steam_historical, "sentiment"),
        (youtube_comments, "sentiment")
    ],
    match_col="matched_missions",
    entity_names=mission_names
)

df_mission_sentiment["polarization_raw"] = df_mission_sentiment["mention_count"] * df_mission_sentiment["sentiment_std"]
df_mission_sentiment["polarization_per_mention"] = df_mission_sentiment["sentiment_std"]
df_mission_sentiment["low_confidence"] = df_mission_sentiment["mention_count"] < MIN_MENTION_THRESHOLD

df_mission_sentiment = df_mission_sentiment.sort_values("mention_count", ascending=False)
df_mission_sentiment.to_csv("../data/processed/missions_sentiment.csv", index=False)
df_mission_sentiment

## Chapter sentiment aggregation

**Two issues were found and fixed during chapter matching** (not shown step-by-step here — see project notes for the full discovery process):
1. "Epilogue Part 1"'s alt-names originally included "John Marston" as an identifying phrase, which caused false-positive matches on any review simply mentioning the character John Marston, unrelated to the Epilogue chapter specifically.
2. Tracking "Epilogue Part 1" and "Epilogue Part 2" as separate entities returned zero matches for Part 2 — reviews don't distinguish between the two halves. They were merged into a single `"Epilogue"` entity below.

In [ ]:
def build_chapter_patterns(df, name_col, alt_col):
    patterns = {}
    for _, row in df.iterrows():
        entity = row[name_col]
        terms = [row[name_col]] + [t.strip() for t in str(row[alt_col]).split(",")]
        terms = [t for t in terms if t and t.lower() != "nan"]
        escaped = [re.escape(t) for t in terms]
        pattern = r"\b(" + "|".join(escaped) + r")\b"
        patterns[entity] = re.compile(pattern, re.IGNORECASE)
    return patterns


chapter_patterns = build_chapter_patterns(ref_chapters, "chapter_name", "chapter_alt_names")

for df, text_col in [(steam_recent, "review_text"), (steam_historical, "review_text"), (youtube_comments, "comment_text")]:
    df["matched_chapters"] = df[text_col].apply(lambda t: find_matches(t, chapter_patterns))

print("Steam recent - rows with chapter match (before fix):", (steam_recent["matched_chapters"].str.len() > 0).sum())

In [ ]:
# Apply the fix: merge the two Epilogue rows into a single entity,
# with corrected alt-names ("Beecher's Hope" + "epilogue" as identifying phrases,
# no longer including "John Marston").
ref_chapters = ref_chapters[~ref_chapters["chapter_name"].isin(["Epilogue Part 1", "Epilogue Part 2"])]

new_row = pd.DataFrame([{
    "chapter_num": 7,
    "chapter_name": "Epilogue",
    "chapter_alt_names": "Beecher's Hope, epilogue"
}])

ref_chapters = pd.concat([ref_chapters, new_row], ignore_index=True)
ref_chapters.to_csv("../data/reference/reference_chapters.csv", index=False)
print(ref_chapters)

In [ ]:
# Re-match with the corrected chapter reference table
chapter_patterns = build_chapter_patterns(ref_chapters, "chapter_name", "chapter_alt_names")

for df, text_col in [(steam_recent, "review_text"), (steam_historical, "review_text"), (youtube_comments, "comment_text")]:
    df["matched_chapters"] = df[text_col].apply(lambda t: find_matches(t, chapter_patterns))

print("Steam recent - rows with chapter match (after fix):", (steam_recent["matched_chapters"].str.len() > 0).sum())

In [ ]:
chapter_names = ref_chapters["chapter_name"].tolist()

df_chapter_sentiment = aggregate_entity_sentiment(
    dfs_with_cols=[
        (steam_recent, "sentiment"),
        (steam_historical, "sentiment"),
        (youtube_comments, "sentiment")
    ],
    match_col="matched_chapters",
    entity_names=chapter_names
)

df_chapter_sentiment["polarization_raw"] = df_chapter_sentiment["mention_count"] * df_chapter_sentiment["sentiment_std"]
df_chapter_sentiment["polarization_per_mention"] = df_chapter_sentiment["sentiment_std"]
df_chapter_sentiment["low_confidence"] = df_chapter_sentiment["mention_count"] < MIN_MENTION_THRESHOLD

df_chapter_sentiment = df_chapter_sentiment.sort_values("mention_count", ascending=False)
df_chapter_sentiment.to_csv("../data/processed/chapters_sentiment.csv", index=False)
df_chapter_sentiment

## Sentiment over time & the epilogue effect

Yearly sentiment trend per game (used for the "legacy inflation" finding), plus the epilogue-mentioning vs. non-epilogue split within RDR2's historical reviews — the strongest single finding in the project (0.89 vs. 0.62 avg sentiment).

In [ ]:
df_hist = steam_historical.copy()
df_hist["date_created"] = pd.to_datetime(df_hist["timestamp_created"], unit="s")
df_hist["year"] = df_hist["date_created"].dt.year
df_hist["year_quarter"] = df_hist["date_created"].dt.to_period("Q").astype(str)

# Yearly sentiment trend per game
yearly_trend = df_hist.groupby(["game", "year"]).agg(
    avg_sentiment=("sentiment", "mean"),
    review_count=("sentiment", "count")
).reset_index()

yearly_trend.to_csv("../data/processed/sentiment_over_time_yearly.csv", index=False)
print(yearly_trend)

In [ ]:
rdr2_hist = df_hist[df_hist["game"] == "RDR2"].copy()

epilogue_mentioned = rdr2_hist[rdr2_hist["matched_chapters"].apply(lambda x: "Epilogue" in x)]
non_epilogue = rdr2_hist[rdr2_hist["matched_chapters"].apply(lambda x: "Epilogue" not in x)]

print("Epilogue-mentioning reviews - avg sentiment:", epilogue_mentioned["sentiment"].mean(), "| count:", len(epilogue_mentioned))
print("Non-epilogue reviews - avg sentiment:", non_epilogue["sentiment"].mean(), "| count:", len(non_epilogue))

# Note: this summary (0.89 vs 0.62, n=59 vs n=2941) was later also exported as a
# row-level CSV (rdr2_epilogue_mention_flag.csv) with an is_epilogue_mention boolean
# column, so Power BI could reproduce this chart natively rather than from a static summary.

## Patch-date correlation (attempted, ultimately excluded from the report)

Tests whether sentiment shifts measurably around major patch/update dates. **Result: sample sizes per patch window (14–18 reviews) were too small to reliably separate genuine patch effects from unrelated, recurring complaints** — this analysis is kept here for documentation of the attempt, but its output (`sentiment_patch_effects.csv`) is not used in the final report.

In [ ]:
# Starter list of major RDR2 PC patches/updates - VERIFY before treating as final
patch_dates = [
    {"patch_date": "2019-11-05", "description": "PC launch"},
    {"patch_date": "2019-12-13", "description": "Post-launch stability patch"},
    {"patch_date": "2020-02-01", "description": "Bounty Hunter update / Frontier Pursuits"},
    {"patch_date": "2021-07-13", "description": "Blood Money update"},
    {"patch_date": "2023-08-15", "description": "Steam Deck compatibility / performance patch"},
]

df_patches = pd.DataFrame(patch_dates)
df_patches["patch_date"] = pd.to_datetime(df_patches["patch_date"])
df_patches.to_csv("../data/reference/reference_patch_dates.csv", index=False)
df_patches

In [ ]:
rdr2_hist = df_hist[df_hist["game"] == "RDR2"].copy()

WINDOW_DAYS = 14

patch_effects = []
for _, patch in df_patches.iterrows():
    window_start = patch["patch_date"]
    window_end = patch["patch_date"] + pd.Timedelta(days=WINDOW_DAYS)

    in_window = rdr2_hist[(rdr2_hist["date_created"] >= window_start) & (rdr2_hist["date_created"] <= window_end)]
    baseline = rdr2_hist[~rdr2_hist.index.isin(in_window.index)]

    patch_effects.append({
        "patch_date": patch["patch_date"],
        "description": patch["description"],
        "reviews_in_window": len(in_window),
        "avg_sentiment_in_window": in_window["sentiment"].mean(),
        "baseline_avg_sentiment": baseline["sentiment"].mean(),
        "sentiment_delta": in_window["sentiment"].mean() - baseline["sentiment"].mean() if len(in_window) > 0 else None
    })

df_patch_effects = pd.DataFrame(patch_effects)
df_patch_effects.to_csv("../data/processed/sentiment_patch_effects.csv", index=False)
df_patch_effects  # reviews_in_window is consistently small (14-18) - see markdown above for why this was excluded from the final report

## RDR2 vs. GTA V comparison metrics

Derives `positive_ratio` and `players_per_1k_reviews` from the Steam/SteamSpy snapshot, and summarizes average Google Trends search interest per game.

In [ ]:
df_comparison = pd.read_csv("../data/processed/comparison_rdr2_gtav.csv")
df_trends = pd.read_csv("../data/processed/comparison_trends.csv")

df_comparison["positive_ratio"] = df_comparison["positive_reviews"] / (df_comparison["positive_reviews"] + df_comparison["negative_reviews"])
df_comparison["players_per_1k_reviews"] = df_comparison["current_players"] / ((df_comparison["positive_reviews"] + df_comparison["negative_reviews"]) / 1000)
df_comparison.to_csv("../data/processed/comparison_rdr2_gtav.csv", index=False)

trends_summary = df_trends[["Red Dead Redemption 2", "GTA V"]].mean().reset_index()
trends_summary.columns = ["game", "avg_search_interest"]
trends_summary.to_csv("../data/processed/comparison_trends_summary.csv", index=False)

print(df_comparison[["game", "positive_ratio", "players_per_1k_reviews"]])
print()
print(trends_summary)

## Summary
All aggregated CSVs are now saved and ready for `03_visuals_export.ipynb` and the Power BI report:
- `characters_sentiment.csv`, `missions_sentiment.csv`, `chapters_sentiment.csv`
- `sentiment_over_time_yearly.csv`
- `sentiment_patch_effects.csv` (excluded from final reporting, kept for documentation)
- `comparison_rdr2_gtav.csv`, `comparison_trends_summary.csv`